**MATH70148 - Probabilistic Generative Models 2025-2026**

# Coursework 2 - Part II


**Instructions:**

You will implement a minimal version of the GPT architecture and train it to generate synthetic text in the style of the Shakespeare's works. Follow this notebook and complete the missing parts according to the given instructions. Register the code you generate in the parts indicated as `YOUR CODE HERE` and your responses (when applicable) by creating a new cell in this notebook. Your submission will be assessed from this notebook only, **no pdf report is needed**.

**Comment on running time:** It is expected that this model is to be trained over at least 5000 epochs to produce text resembling the stucture of a play, and you are strongly encouraged to run it on Google Colab (where the 5000 epochs should take around 6 minutes if using a T4 runtime type) or a computer with a sufficiently capable GPU. For reference, if ran on a regular MPS-enabled Apple M3 Pro Macbook Pro, this code will take ~10 minutes, and ~5.5 minutes if using a NVIDIA GeForce GTX 1080 GPU.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## 1. Download the corpus and tokenise it

In [ ]:
# If using Colab, consider:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
# Load the provided dataset
path = '/content/drive/MyDrive/Imperial College London/PGM/CW2/shakespeare.txt'

In [ ]:
# Read the entire Shakespeare corpus into memory
# We explicitly set UTF-8 encoding to avoid platform-dependent issues.
with open(path, 'r', encoding='utf-8') as file:
    text = file.read()

# Basic corpus statistics
num_chars = len(text)

print("=== Corpus Statistics ===")
print(f"Total number of characters: {num_chars:,}")
print(f"First 200 characters preview:\n")
print(text[:200])

=== Corpus Statistics ===
Total number of characters: 1,115,394
First 200 characters preview:

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


In [ ]:
# Extract the set of unique characters appearing in the text
# `set(text)` removes duplicates
# `sorted(...)` ensures a deterministic ordering (important for reproducibility)
chars = sorted(list(set(text)))

# Size of the vocabulary (number of unique characters)
vocab_size = len(chars)

# Display results
print("Unique characters (sorted):")
print("".join(chars))  # Join list into a readable string

print("\nVocabulary size:")
print(f"{vocab_size} unique characters")

Unique characters (sorted):

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz

Vocabulary size:
65 unique characters


In [ ]:
# --------------------------------------------------
# Character-level tokenizer
# --------------------------------------------------

# NOTE:
# The model will operate at the character level, meaning that the tokens are characters (and not words)

# Create a mapping from character to integer index
# This is our "string-to-index" (stoi) dictionary.
stoi = {char: idx for idx, char in enumerate(chars)}

# Create the inverse mapping: index to character
# This is needed to convert model outputs back to text.
itos = {idx: char for idx, char in enumerate(chars)}


def encode(s):
    """
    Convert a string into a list of integer token IDs.

    Args:
        s (str): Input string.

    Returns:
        List[int]: List of corresponding character indices.
    """
    return [stoi[char] for char in s]


def decode(indices):
    """
    Convert a list of integer token IDs back into a string.

    Args:
        indices (List[int]): List of character indices.

    Returns:
        str: Decoded string.
    """
    return ''.join(itos[idx] for idx in indices)


In [ ]:
# Convert the entire corpus into a PyTorch tensor of token IDs.
# We use dtype=torch.long because embedding layers expect integer (int64) indices.
data = torch.tensor(encode(text), dtype=torch.long)

# Inspect the shape of the resulting tensor
print("Data tensor shape:", data.shape)

Data tensor shape: torch.Size([1115394])


In [ ]:
# Display a short excerpt of the original text and its corresponding
# integer (token ID) representation to illustrate the encoding process.

N = 50

print(f"Text containing the first {N} characters:")
print("-" * 50)
print(text[:N])

print("\n" + "-" * 50)
print("Its representation as a PyTorch tensor:\n")
print(data[:N])

Text containing the first 50 characters:
--------------------------------------------------
First Citizen:
Before we proceed any further, hear

--------------------------------------------------
Its representation as a PyTorch tensor:

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56])


## 2. Dataset split

In [ ]:
# Split the dataset into training (90%) and validation (10%) sets.
# We consider a sequential split to preserve temporal structure.

n = int(0.9 * len(data))  # 90% for training
train_data = data[:n]
val_data = data[n:]

print(f"Training corpus size: {len(train_data):,} characters "
      f"({train_data.shape[0] / data.shape[0]:.2%} of total)")

print(f"Validation corpus size: {len(val_data):,} characters "
      f"({val_data.shape[0] / data.shape[0]:.2%} of total)")

Training corpus size: 1,003,854 characters (90.00% of total)
Validation corpus size: 111,540 characters (10.00% of total)


### On the data–label mechanics

- We access the data through *context windows*: contiguous blocks of text in numerical (token) representation of size `block_size`.

- The model is self-supervised. That is, we train it so that, given a sequence of tokens ${x}_{i:j}$, it predicts the next token $x_{j+1}$.

- The labels are obtained from the same context window by shifting it with an offset of 1 position.

In [ ]:
# EXAMPLE:

# Given a context window of size 7,
# we construct an input–target pair (X, Y),
# where Y is simply X shifted by one position.

block_size = 7

print("A context window (X, Y) looks like:\n")

# Input sequence (context)
X = data[:block_size]
print(f"X (token IDs): {[x.item() for x in X]}")
print(f"--> decode(X): {decode([x.item() for x in X])}")

print("-" * 40)

# Target sequence (shifted by one position)
Y = data[1:block_size + 1]
print(f"Y (token IDs): {[y.item() for y in Y]}")
print(f"--> decode(Y): {decode([y.item() for y in Y])}")

A context window (X, Y) looks like:

X (token IDs): [18, 47, 56, 57, 58, 1, 15]
--> decode(X): First C
----------------------------------------
Y (token IDs): [47, 56, 57, 58, 1, 15, 47]
--> decode(Y): irst Ci


In [ ]:
# EXAMPLE (continued):

# However, within each context window we use the information
# in an autoregressive manner, generating multiple training
# examples from a single block.

x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    context = x[:t + 1]      # Tokens available at time step t
    target = y[t]            # Next token to predict

    print(f"When the input is {context.tolist()}, the target is: {target.item()}")

When the input is [18], the target is: 47
When the input is [18, 47], the target is: 56
When the input is [18, 47, 56], the target is: 57
When the input is [18, 47, 56, 57], the target is: 58
When the input is [18, 47, 56, 57, 58], the target is: 1
When the input is [18, 47, 56, 57, 58, 1], the target is: 15
When the input is [18, 47, 56, 57, 58, 1, 15], the target is: 47


Therefore, each context window generates a number of training observations equal to its length.

We now turn to *batches*, which allows us to process multiple context windows in parallel. The key point is that each context window is independent: there is no computation that occurs across windows, only parallel computation within them. In other words, the autoregressive sequences corresponding to different context windows are not mixed. See the following example.

In [ ]:
torch.manual_seed(1337)

batch_size = 4
block_size = 8  # maximum context length considered for prediction
# These hyperparameters will be redefined for the final training phase


def get_batch(split):
    """
    Generate a small batch of input (x) and target (y) sequences.

    Each batch contains `batch_size` independent context windows
    of length `block_size`, sampled randomly from the dataset.
    """
    data_split = train_data if split == 'train' else val_data

    # Random starting positions for each context window
    ix = torch.randint(len(data_split) - block_size, (batch_size,))

    # Construct input and target batches
    x = torch.stack([data_split[i:i + block_size] for i in ix])
    y = torch.stack([data_split[i + 1:i + block_size + 1] for i in ix])

    return x, y


# Sample a training batch
xb, yb = get_batch('train')

print("Inputs:")
print("Shape:", xb.shape)   # (batch_size, block_size)
print(xb)

print("\nTargets:")
print("Shape:", yb.shape)   # (batch_size, block_size)
print(yb)

Inputs:
Shape: torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])

Targets:
Shape: torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


### Questions
 - Q1 (5% marks): Explain why autoregressive language modeling is considered self-supervised. What are the inputs and labels in this setup?
 - Q2 (5% marks): From a context window of size $T$, how many training examples are generated? Explain why.

Q1: Autoregressive language modeling is considered self-supervised because it extracts supervision signals directly from the raw data, reducing the need for manual human labeling in each input text. In this setup, the inputs are defined as a sequence of preceding tokens, $(x_1, x_2, \dots, x_n)$, while the label is the immediate subsequent token, $x_{n+1}$, which serves as the ground truth. During training, the model generates a probabilistic prediction for the next element. The discrepancy between this prediction and the true token constitutes a loss (typically cross-entropy), providing the necessary signal to update the model's parameters.

Q2: For a single sequence $(x_{1}, x_{2}, \ldots x_{T}) $with size $T$, we may have $T$ training samples. As it is an autogression model, we may start predicting $x_{2}$ given $x_{1}$, and finally predict $x_{T+1}$ given $x_{1}, x_{2}, \ldots x_{T}$.

## 3. Self-attention

- Q3 (15% marks): Implement the missing parts in the attention head class. Recall that in this model you need to implement masked attention.

In [ ]:
# Embedding dimension (model "width"): each token is represented as a vector in R^{n_embd}
n_embd = 64
dropout = 0.0


class Head(nn.Module):
    """
    A single self-attention head (causal / autoregressive).

    Given an input sequence of token embeddings x of shape (B, T, n_embd),
    this module produces an output of shape (B, T, head_size) by:
      1) projecting x into queries, keys, and values
      2) computing attention weights via scaled dot-product attention
      3) applying a causal mask so tokens cannot attend to the future
      4) computing a weighted sum of value vectors
    """

    def __init__(self, head_size: int):
        super().__init__()


        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        # -------------------------------------------------------- #
        self.key = nn.Linear(n_embd, head_size, bias=False) #(n_embd, dk)
        self.query = nn.Linear(n_embd, head_size, bias=False) #(n_embd, dk)
        self.value = nn.Linear(n_embd, head_size, bias=False) # (n_embd, dv)


        # Causal mask (lower-triangular): prevents attending to future positions.
        # register_buffer makes it part of the module state (moves with .to(device))
        # but it is not a learnable parameter.
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

        # Dropout applied to the attention weights (common regularization in Transformers)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:


        """
        Args:
            x: (B, T, n_embd) input token embeddings

        Returns:
            out: (B, T, head_size) attention output for this head
        """
        T = x.shape[1]

        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        # -------------------------------------------------------- #
        k = self.key(x)   # (B, T, dk) and key is a network (matrix)
        q = self.query(x) # (B, T, dk) and query is a network (matrix)
        v = self.value(x) # (B, T, dv) and value is a network (matrix)

        # Compute attention weights
        weights = q @ k.transpose(-2, -1) * (k.shape[-1]**-0.5) # (B, T, dk) @ (B, dk, T) -> (B, T, T)

        # Apply causal mask (prevents tokens from attending to future tokens)
        weights = weights.masked_fill(self.tril[:T, :T] == 0, float('-inf'))

        # Softmax to get attention probabilities
        weights = F.softmax(weights, dim=-1)

        # Apply dropout to attention weights
        weights = self.dropout(weights)

        # Perform the weighted aggregation of the values
        out = weights @ v  # (B, T, T) @ (B, T, dv) -> (B, T, dv)

        return out

### Questions
 - Q3 (continued): Explain why the mask is needed.  

In autoregressive language modeling, a causal mask is necessary to prevent 'information leakage.' Since the entire sequence is available during training, without a mask, the model would be able to 'cheat' by directly attending to future tokens that it is supposed to predict. To prevent this, the mask is applied to the attention scores (the dot product of current queries and future keys) by setting them to negative infinity ($-\infty$). This ensures that after the softmax operation, the attention weights for future positions become zero, forcing the model to rely solely on the preceding context to make its predictions.

## 4. Multi-head Attention

- Q4 (10% marks): Implement the missing parts in the multihead attention class.

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multiple self-attention heads running in parallel.

    Each head independently computes attention over the same input.
    Their outputs are concatenated and linearly projected back to the
    embedding dimension (n_embd).
    """

    def __init__(self, num_heads: int, head_size: int):
        super().__init__()

        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        # -------------------------------------------------------- #
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)]) #(num_head, n_embd * head_size)
        self.proj = nn.Linear(head_size * num_heads, n_embd)  # (head_size * num_head, n_embd)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, T, n_embd)

        Returns:
            out: (B, T, n_embd)
        """
        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        # -------------------------------------------------------- #
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)

        return out

## 5. MLP stage

- Q5 (7.5% marks): Implement the missing parts in the FeedForward attention class.

In [ ]:
class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network (single-hidden layer MLP)
    This module is applied independently to each position in the sequence.

    Input:  (B, T, n_embd)
    Output: (B, T, n_embd)
    """

    def __init__(self, n_embd: int):
        super().__init__()

        # The Transformer FFN consists of two linear layers with a non-linearity in between.
        # The inner layer expands the dimensionality by a factor of 4,
        # then projects back to the original embedding size.

        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        # -------------------------------------------------------- #
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd)
        )



    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Applies the feed-forward network independently at each time step.
        No interaction occurs across sequence positions.
        """
        return self.net(x)

### Questions
 - Q6 (10% marks): Explain the relationship between the hyperparameters `n_head` and `head_size` in the `MultiHeadAttention` class. Consider their role within the decoder block (i.e., attention + feed-forward).

We state the relationship is `n_embd = n_head * head_size` which defines the decomposition of the embedding space into multiple parallel subspaces. This allows the multi-head attention mechanism to process tokens in 'batches' of dimensions, capturing diverse linguistic aspects simultaneously. Within the decoder block, this identity ensures that the attention output maintains the original `n_embd` dimension, which is essential for residual connections and for providing a compatible input to the subsequent feed-forward layer.

## 6. Decoder block

- Q7 (10% marks): Implement the missing parts in the DecoderBlock class.

In [ ]:
class DecoderBlock(nn.Module):
    """
    Transformer decoder block: COMMUNICATION followed by COMPUTATION.

    - Communication: masked multi-head self-attention (tokens exchange information)
    - Computation: position-wise feed-forward network (nonlinear transformation per token)

    Your implementation should use the Pre-LayerNorm (Pre-LN) variant:
    LayerNorm is applied before each sublayer, and residual (skip) connections are applied around each sublayer.
    """

    def __init__(self, n_embd: int, n_head: int):
        # n_embd: embedding dimension, n_head: number of attention heads

        super().__init__()

        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        # -------------------------------------------------------- #

        self.mha = MultiHeadAttention(n_head, n_embd // n_head)
        self.ff = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)




    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, T, n_embd) input embeddings

        Returns:
            x: (B, T, n_embd) output after attention + feed-forward, with residuals
        """
        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        # -------------------------------------------------------- #
        x = x + self.mha(self.ln1(x))
        x = x + self.ff(self.ln2(x))

        return x

## 7. GPT Model: Putting it all together

- Q8 (17.5% marks): Implement the missing computation of the loss in the `GPTLanguageModel` class and the `generate` method to sample tokens that autoregressively extend a sequence.

In [ ]:
class GPTLanguageModel(nn.Module):
    """
    A minimal GPT-style (decoder-only) Transformer language model.

    Components:
      - Token embeddings: map token IDs -> vectors in R^{n_embd}
      - Positional embeddings: encode token position within the context window
      - A stack of decoder blocks: masked self-attention + feed-forward (+ residuals)
      - Final layer norm + linear head: produce logits over the vocabulary

    Input:
      idx: (B, T) integer token IDs

    Output:
      logits: (B, T, vocab_size)
      loss: scalar cross-entropy loss (if targets are provided)
    """

    def __init__(self):
        super().__init__()

        # Token and position embeddings
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # Stack of decoder blocks
        self.blocks = nn.Sequential(
            *[DecoderBlock(n_embd, n_head=n_head) for _ in range(n_layer)]
        )

        # Final normalization and language modeling head
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

        # Weight initialization (close to the GPT/Transformer convention)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        """
        Args:
            idx: (B, T) token IDs
            targets: (B, T) token IDs (optional)

        Returns:
            logits: (B, T, vocab_size)
            loss: scalar loss if targets is provided, else None
        """
        B, T = idx.shape

        # Token embeddings: (B, T, n_embd)
        tok_emb = self.token_embedding_table(idx)

        # Positional embeddings: (T, n_embd)
        # We create positions [0, 1, ..., T-1] on the correct device.
        pos = torch.arange(T, device=idx.device)
        pos_emb = self.position_embedding_table(pos)

        # Add token and positional information (broadcast pos_emb over batch dimension)
        x = tok_emb + pos_emb  # (B, T, n_embd)

        # Decoder blocks + final layer norm
        x = self.blocks(x)     # (B, T, n_embd)
        x = self.ln_f(x)       # (B, T, n_embd)

        # Convert to vocabulary logits
        logits = self.lm_head(x)  # (B, T, vocab_size)

        # -------------------------------------------------------- #
        # --------------------[YOUR CODE HERE]-------------------- #
        # --------(computation of the cross entropy loss)--------- #
        loss = None
        if targets is not None:
            # Reshape logits to (B*T, vocab_size) and targets to (B*T)
            # This is necessary for F.cross_entropy which expects (N, C) and (N) respectively
            B, T, C = logits.shape
            logits_flat = logits.view(B * T, C)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)

        return logits, loss

    @torch.no_grad()
    def generate(self, idx: torch.Tensor, max_new_tokens: int, temperature: float = 1.0):
        """
        Autoregressively sample new tokens.

        Args:
            idx: (B, T) current context (token IDs)
            max_new_tokens: number of tokens to generate
            temperature: > 0, controls randomness (lower is more deterministic)

        Returns:
            idx: (B, T + max_new_tokens) with generated tokens appended
        """
        self.eval()

        for _ in range(max_new_tokens):

            # -------------------------------------------------------- #
            # --------------------[YOUR CODE HERE]-------------------- #
            # -------------------------------------------------------- #

            # Crop idx to the last block_size tokens
            # This is to ensure we don't feed an infinitely growing context to the model
            idx_cond = idx[:, -block_size:]

            # Get predictions (logits) for the next token
            logits, _ = self(idx_cond)  # (B, T_cond, vocab_size)

            # Focus only on the last time step (the prediction for the next token)
            logits = logits[:, -1, :]  # (B, vocab_size)

            # Apply temperature to control randomness
            logits = logits / temperature

            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1)  # (B, vocab_size)

            # Sample the next token from the distribution
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1)

            # Append the sampled token to the current sequence
            idx = torch.cat((idx, idx_next), dim=1)

        return idx

## 8. Training specs

In [ ]:
# Define hyperparameters (global variables). In a larger project, it is better
# practice to store these in a config object or a dataclass to avoid relying on globals.

# -------------------------------------------------------------------
# Choose the device we want to run on (CUDA > MPS > CPU)
# -------------------------------------------------------------------
if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Using device:", device)

# -------------------------------------------------------------------
# Training hyperparameters
# -------------------------------------------------------------------
batch_size = 16      # B: number of independent sequences (context windows) processed in parallel
block_size = 128     # T: maximum context length (number of tokens used to predict the next token)
max_iters = 5000     # total number of optimization steps
eval_interval = 500  # evaluate the model every eval_interval steps
eval_iters = 200     # number of batches used to estimate train/val loss during evaluation
learning_rate = 3e-4 # a common default for small GPT-style models (with AdamW)

# -------------------------------------------------------------------
# Model hyperparameters
# -------------------------------------------------------------------
n_embd = 384   # embedding dimension (must satisfy: n_head * head_size = n_embd)
n_head = 6     # number of attention heads (so head_size = n_embd // n_head = 64 here)
n_layer = 6    # number of decoder blocks
dropout = 0.2  # dropout probability (regularization)

# -------------------------------------------------------------------
# Build model and optimizer
# -------------------------------------------------------------------
model = GPTLanguageModel().to(device)

# Print the number of parameters (in millions)
num_params = sum(p.numel() for p in model.parameters())
print(f"Number of model parameters: {num_params/1e6:.2f} million")

# AdamW is the standard optimizer for Transformer language models
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

Using device: cuda:0
Number of model parameters: 10.74 million


We now define a function to generate batches of sequences from our training or validation corpus. We will also define a function to obtain estimates of the loss function.

In [ ]:
# ------------------------------------------------------------
# Mini DataLoader: sample random context windows from the corpus
# ------------------------------------------------------------
def get_batch(split: str):
    """
    Generate a batch of input (x) and target (y) sequences.

    Args:
        split: 'train' or 'val'

    Returns:
        x: (B, T) input token IDs
        y: (B, T) target token IDs (shifted by one position)
    """
    data_split = train_data if split == 'train' else val_data

    # Random starting positions for each sequence in the batch
    ix = torch.randint(len(data_split) - block_size, (batch_size,))

    # Build input and target sequences
    x = torch.stack([data_split[i:i + block_size] for i in ix])
    y = torch.stack([data_split[i + 1:i + block_size + 1] for i in ix])

    # Move to the selected device (CPU / GPU / MPS)
    x, y = x.to(device), y.to(device)

    return x, y


# ------------------------------------------------------------
# Estimate average loss on train and validation sets
# ------------------------------------------------------------
@torch.no_grad()
def estimate_loss():
    """
    Compute an average loss estimate over `eval_iters` batches
    for both training and validation splits.
    """
    out = {}
    model.eval()  # disable dropout, use evaluation mode

    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)

        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()

        out[split] = losses.mean()

    model.train()  # return to training mode
    return out

First sanity check: make sure the sampled batches have the correct dimensions.

In [ ]:
# Verify that the sampled batches have the expected dimensions
xb, yb = get_batch('train')

print("Input batch shape :", xb.shape)  # Expected: (batch_size, block_size)
print("Target batch shape:", yb.shape)  # Expected: (batch_size, block_size)

Input batch shape : torch.Size([16, 128])
Target batch shape: torch.Size([16, 128])


Second sanity check: verify that the data flows correctly through the model.

In [ ]:
# Verify that the model's forward pass runs without errors
# and produces logits of the expected shape

logits, _ = model(xb)

print("Logits shape:", logits.shape)
# Expected: (batch_size, block_size, vocab_size)

Logits shape: torch.Size([16, 128, 65])


## 9. Train

In [ ]:
# Use your CID as the random seed (for reproducibility)
torch.manual_seed(106046958)

# ------------------------------------------------------------
# Training loop
# ------------------------------------------------------------
for step in range(max_iters):

    # Periodically estimate loss on both training and validation splits
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # Sample a batch of training data
    xb, yb = get_batch('train')

    # Forward pass: compute logits and loss
    logits, loss = model(xb, yb)

    # Backward pass + parameter update
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

step 0: train loss 4.2688, val loss 4.2707
step 500: train loss 2.1028, val loss 2.1573
step 1000: train loss 1.7088, val loss 1.8573
step 1500: train loss 1.5613, val loss 1.7346
step 2000: train loss 1.4728, val loss 1.6593
step 2500: train loss 1.4130, val loss 1.6157
step 3000: train loss 1.3683, val loss 1.5860
step 3500: train loss 1.3422, val loss 1.5574
step 4000: train loss 1.3103, val loss 1.5391
step 4500: train loss 1.2937, val loss 1.5299
step 4999: train loss 1.2698, val loss 1.5168


## 10. Generate text sequences with the trained model

In [ ]:
# ------------------------------------------------------------
# Generate text using the trained model
# ------------------------------------------------------------

# Start from a minimal context (single token with ID 0)
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# Generate 1500 new tokens autoregressively
generated_tokens = model.generate(context, max_new_tokens=1500)

# Decode token IDs back into characters
print(decode(generated_tokens[0].tolist()))


# ------------------------------------------------------------
# Optionally, write generated text to a file
# ------------------------------------------------------------

# open('output.txt', 'w').write(
#     decode(model.generate(context, max_new_tokens=10000)[0].tolist())
# )


Life:
though in the trouble born an eaged, not that dales
Shall have thee that veing maid
from let in the provid your mother, but one I
On the one the deliver, you marched am here well.
Speak how come on your green?

Lord:
I pray him, I am a very captain and good view;
For I say, if grither we have been, is better.

LEONTES:
Of my wife
The contrain, 'But's to the writer reign vile to thee?

ABHOS:
But that have worm what now hasty that my pardon
Here lies and therefore way; therefore and not the trun.

HENRY BOLINGBROKE:
But, my lord, it was they brother metitlement,
By Knowing of the grooners, uncle, and to thief
Like oft the transman-carr'd, we will smild here with
strong-factify, and doing thee avoid, ask, And
That state as not such a life
For more and shall confirm soul are for love,
Her more die on himself from his captain.

Second Senator:
My Lord, decry Captua,
What they do being butting on my worship
And when two he with his copesity or stream;
But I shall mark thee weakness l

### Questions
 - Q9 (20% marks): Comment on the generated text.
   - Does it learn local syntax?
   - Does it maintain long-term coherence?
   - How does the temperature affect the generated text?
   - What architectural limitation explains its behavior?

Yes, the model learns local syntax. It understands how words should be ordered on a surface level. For example, it knows that a verb usually follows a subject and a noun often follows an adjective. This creates the illusion of structure (Subject-Verb-Object), even though the meaning doesn't make sense.

No, the model fails to maintain long-term coherence. While it effectively captures local dependencies (the relationship between neighboring tokens), it struggles to integrate global context from the entire text. This limitation prevents the model from generating a logically consistent narrative, as it prioritizes immediate statistical patterns over long-range semantic structure

In terms of Temperature ($T$), a higher value increases the model's output stochasticity by flattening the logit distribution. Since logits are scaled as `logits / T` before being passed to the Softmax function, a high $T$ reduces the relative gap between the largest and smallest values. This results in a more uniform (non-peaked) probability distribution, preventing the model from becoming overly confident in a single 'top' token and allowing for more diverse and creative predictions during the sampling process.

The restricted dimension of our embedding layer significantly impacts the model's performance. Compared to GPT-3's 12,288-dimensional embeddings, our model utilizes only 64 dimensions, which severely limits its capacity to capture nuanced textual characteristics and represent complex information. Furthermore, the insufficient number of attention heads prevents the model from effectively resolving syntactic and semantic dependencies, such as distinguishing between subjects and objects, leading to the logical inconsistencies observed in the generated text.